# 2. Simulate a small company
a) Connect python to gemini, very important that you place the api key in .env and gitignore it


In [1]:
from dotenv import load_dotenv
from google import genai

client = genai.Client()


b) Use gemini to simulate 20 data points in json format containing the following fields: first_name, last_name, phone_number, email, department, salary, title. See if you can prompt to direct the LLM output to have swedish names, phone numbers in swedish format (+46 731 29 52), departments (IT, HR, marketing, sales), reasonable salary (you might need to check some swedish statistics on salaries) and corresponding titles within these departments.


In [2]:
def ask_llm(prompt):

    response = client.models.generate_content(
        model="gemini-2.5-flash", 
        contents=prompt
    )

    return response.text

response = ask_llm("""
    Du är en expert på statistik inom svenska företag och svensk arbetsmarknad.
    Generera 20 datapunkter i json format (inga markdowns) med följande fält:
    first_name, last_name, phone_number, email, department, salary, title.
    Använd endast svenska format och värden. Departments skall vara 
    IT, HR, marketing, sales och titlarna vara relaterade till dessa departments.
    
    Exempel:
            {
                "first_name": "Birgitta",
                "last_name": "Andersson",
                "phone_number": +46 731 29 52,
                "email": "andersson.birgitta@gmail.com",
                "department": "HR",
                "salary": 36000,
                "title": "personaladministratör"
            }               
                   
                   """)
print(response)

[
    {
        "first_name": "Anna",
        "last_name": "Andersson",
        "phone_number": "+46 70 123 45 67",
        "email": "anna.andersson@foretag.se",
        "department": "HR",
        "salary": 36000,
        "title": "Personaladministratör"
    },
    {
        "first_name": "Johan",
        "last_name": "Karlsson",
        "phone_number": "+46 70 987 65 43",
        "email": "johan.karlsson@itbolaget.se",
        "department": "IT",
        "salary": 48000,
        "title": "Systemutvecklare"
    },
    {
        "first_name": "Maria",
        "last_name": "Nilsson",
        "phone_number": "+46 73 555 12 34",
        "email": "maria.nilsson@marknad.se",
        "department": "marketing",
        "salary": 42000,
        "title": "Digital Marknadsförare"
    },
    {
        "first_name": "Erik",
        "last_name": "Lindberg",
        "phone_number": "+46 76 111 22 33",
        "email": "erik.lindberg@saljbolaget.se",
        "department": "sales",
        "salary": 3


c) Now use pydantic to validate this json and put in proper schema that the fields should follow. You might need to do some processing such as removing backticks and maybe loading json data into a list with json.loads(). Also make sure that only correctly validated data should be stored.


In [3]:
from pydantic import BaseModel, Field, EmailStr
import json

# basemodel for a staff object
class Staff(BaseModel):
    first_name: str 
    last_name: str 
    phone_number: str 
    email: EmailStr 
    department: str 
    salary: int 
    title: str
# basemodel for a list of staff
class Roster(BaseModel):
    employee_directory: list[Staff]
# validate and deserialize to python objects    
roster = Roster.model_validate({"employee_directory": json.loads(response)})

roster.employee_directory 

[Staff(first_name='Anna', last_name='Andersson', phone_number='+46 70 123 45 67', email='anna.andersson@foretag.se', department='HR', salary=36000, title='Personaladministratör'),
 Staff(first_name='Johan', last_name='Karlsson', phone_number='+46 70 987 65 43', email='johan.karlsson@itbolaget.se', department='IT', salary=48000, title='Systemutvecklare'),
 Staff(first_name='Maria', last_name='Nilsson', phone_number='+46 73 555 12 34', email='maria.nilsson@marknad.se', department='marketing', salary=42000, title='Digital Marknadsförare'),
 Staff(first_name='Erik', last_name='Lindberg', phone_number='+46 76 111 22 33', email='erik.lindberg@saljbolaget.se', department='sales', salary=39000, title='Säljare'),
 Staff(first_name='Sara', last_name='Johansson', phone_number='+46 72 345 67 89', email='sara.johansson@techsupport.se', department='IT', salary=38500, title='IT-supporttekniker'),
 Staff(first_name='Per', last_name='Svensson', phone_number='+46 70 234 56 78', email='per.svensson@rekry


d) Write this json data to a folder called output_data.


In [4]:
import os

# create a folder if not exist
os.makedirs("output_data", exist_ok=True)

# save data to json filter in folder
output_path = os.path.join("output_data", "employee_directory.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(roster.model_dump(), f, ensure_ascii=False, indent=4)


e) Use pandas to read the data as dataframe


In [5]:
import pandas as pd

df = pd.DataFrame(
    [
        roster.model_dump(include={"first_name", "last_name","phone_number", "email", "department", "salary", "title"})
             for roster in roster.employee_directory
    ]
)
df

,first_name,last_name,phone_number,email,department,salary,title
0,Anna,Andersson,+46 70 123 45 67,anna.andersson@foretag.se,HR,36000,Personaladministratör
1,Johan,Karlsson,+46 70 987 65 43,johan.karlsson@itbolaget.se,IT,48000,Systemutvecklare
2,Maria,Nilsson,+46 73 555 12 34,maria.nilsson@marknad.se,marketing,42000,Digital Marknadsförare
3,Erik,Lindberg,+46 76 111 22 33,erik.lindberg@saljbolaget.se,sales,39000,Säljare
4,Sara,Johansson,+46 72 345 67 89,sara.johansson@techsupport.se,IT,38500,IT-supporttekniker
5,Per,Svensson,+46 70 234 56 78,per.svensson@rekrytering.se,HR,41000,Rekryterare
6,Lena,Gustavsson,+46 73 888 77 66,lena.gustavsson@kampanj.se,marketing,37000,Marknadskoordinator
7,Daniel,Olsson,+46 76 444 55 66,daniel.olsson@account.se,sales,51000,Account Manager
8,Sofia,Eriksson,+46 72 765 43 21,sofia.eriksson@natverk.se,IT,45000,Nätverkstekniker
9,Magnus,Larsson,+46 70 876 54 32,magnus.larsson@hrkonsult.se,HR,47000,HR-specialist



f) Write a csv file to your output_data


In [6]:
df.to_csv("output_data/employee_directory.csv", index=False)

In [8]:
df.head()

,first_name,last_name,phone_number,email,department,salary,title
0,Anna,Andersson,+46 70 123 45 67,anna.andersson@foretag.se,HR,36000,Personaladministratör
1,Johan,Karlsson,+46 70 987 65 43,johan.karlsson@itbolaget.se,IT,48000,Systemutvecklare
2,Maria,Nilsson,+46 73 555 12 34,maria.nilsson@marknad.se,marketing,42000,Digital Marknadsförare
3,Erik,Lindberg,+46 76 111 22 33,erik.lindberg@saljbolaget.se,sales,39000,Säljare
4,Sara,Johansson,+46 72 345 67 89,sara.johansson@techsupport.se,IT,38500,IT-supporttekniker



g) Load this data into a staging layer and store this into a table called employees.



In [10]:
import dlt

# create a table, replace data in table
@dlt.resource(write_disposition="replace", table_name="employees")

def load_data():
    yield df

pipeline = dlt.pipeline(
    pipeline_name="employees",
    destination="duckdb",
    dataset_name="staging"
)
load_info = pipeline.run(load_data())
print(load_info)

Pipeline employees load step completed in 0.15 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\metal\Skola\course\ai_engineering_stefan_lundberg\exercises\exercise_3\employees.duckdb location to store data
Load package 1757921610.6155531 is LOADED and contains no failed jobs



h) Use gemini to simulate departments data. There should be same departments as those you had in task b. Also add a description field and a contact person.


In [17]:
from typing import Literal

class Department(BaseModel):
    department: BaseModel
    contact: str
    description: str

class Departments(BaseModel):
    departments_list: list[Department]

responses = client.models.generate_content(
    model = "gemini-2.5-pro",

    contents = """Generera departments data. Departments skall vara 
    IT, HR, marketing, sales och till varje respektive department skall det finnas 
    en kontaktperson och en kort description.
    Använd endast svenska format och värden (förutom department).
    Exempel:
            {
                "department": "HR",
                "contact": "Sven Andersson",
                "description": "Humar relations avdelning osv."
            }
    """,
    config = {"response_mime_type": "application/json", "response_schema": list[Department]}
)

print(responses)


#     response = client.models.generate_content(
#     model = "gemini-2.5-pro",
#     contents = """List 50 apartments and houses in Sweden with their monthly_fee, their price, living area, 
#     number of rooms, address, type if it is apartment or house. 
#     All currencies are in SEK.""",
#     config = {"response_mime_type": "application/json", "response_schema": list[Home]}
#     )


# homes = response.parsed
# homes



ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': '* GenerateContentRequest.generation_config.response_schema.items.properties["department"].properties: should be non-empty for OBJECT type\n', 'status': 'INVALID_ARGUMENT'}}


i) Add a departments table in your duckdb database under staging layer to store this data.